In [1]:
R.version.string

[1] "R version 4.5.2 (2025-10-31)"

In [2]:
setwd('../data/graphs/better_graph')

In [3]:
library(statnet)
library(dplyr)
library(intergraph)
library(igraph)
library(ergm.count)
library(Matrix)

Loading required package: tergm

Loading required package: ergm

Loading required package: network




‘network’ 1.19.0 (2024-12-08), part of the Statnet Project
* ‘news(package="network")’ for changes since last version
* ‘citation("network")’ for citation information
* ‘https://statnet.org’ for help, support, and other information



‘ergm’ 4.10.1 (2025-08-26), part of the Statnet Project
* ‘news(package="ergm")’ for changes since last version
* ‘citation("ergm")’ for citation information
* ‘https://statnet.org’ for help, support, and other information


‘ergm’ 4 is a major update that introduces some backwards-incompatible
changes. Please type ‘news(package="ergm")’ for a list of major
changes.


Loading required package: networkDynamic


‘networkDynamic’ 0.11.5 (2024-11-21), part of the Statnet Project
* ‘news(package="networkDynamic")’ for changes since last version
* ‘citation("networkDynamic")’ for citation information
* ‘https://statnet.org’ for help, support, and other information


Registered S3 method overwritten by 'tergm':
  method                   from
  simulate_formula

In [4]:
nodes <- read.csv("./nodes.csv", stringsAsFactors = FALSE)
edges <- read.csv("./edges.csv", stringsAsFactors = FALSE)

In [5]:
# We assume:
# - nodes has at least: artist_mbid, artist_name, window_years or years_active
# - edges has at least: u, v (both artist_mbid), and any optional edge attributes

if (!all(c("u", "v") %in% names(edges))) {
  stop("edges must have columns 'u' and 'v' (artist_mbid).")
}
if (!"artist_mbid" %in% names(nodes)) {
  stop("nodes must have column 'artist_mbid'.")
}
if (!"artist_name" %in% names(nodes)) {
  stop("nodes must have column 'artist_name'.")
}

zscore <- function(x) {
  x <- as.numeric(x)
  if (length(x) == 0) return(x)
  if (all(is.na(x))) return(x)
  mu <- mean(x, na.rm = TRUE)
  sd_val <- stats::sd(x, na.rm = TRUE)
  if (is.na(sd_val) || sd_val == 0) return(x - mu)
  (x - mu) / sd_val
}

# Canonical vertex key = artist_mbid
nodes$mbid  <- as.character(nodes$artist_mbid)

# Make sure edge endpoints are character mbids
edges$u <- as.character(edges$u)
edges$v <- as.character(edges$v)

# We'll work on a copy of nodes for graph vertices
nodes_df <- nodes

## ------------------------------------------------------------------
## 3) Basic type normalization
## ------------------------------------------------------------------

# Character columns that should always be character if present
char_cols <- c(
  "name", "mbid", "artist_mbid", "artist_name",
  "primary_genre", "all_genres_str",
  "primary_label", "all_labels_str",
  "primary_role", "all_roles_str",
  "artist_country", "artist_region_city",
  "debut_date", "window_cutoff_date"
)
char_cols <- intersect(char_cols, names(nodes_df))
for (cc in char_cols) {
  nodes_df[[cc]] <- as.character(nodes_df[[cc]])
}

# Numeric columns that should always be numeric if present
num_cols <- c(
  "window_years", "years_active",
  "releases_total", "releases_per_year",
  "tracks_total", "avg_days_between_releases",
  "release_velocity_releases_per_day",
  "release_velocity_releases_per_year",
  "gap_median_days", "gap_std_days",
  "max_dry_spell_days", "front_loading_index",
  "collab_track_rate", "unique_collaborator_count",
  "label_diversity_count", "label_churn", "label_hhi",
  "duration_ms_mean", "duration_ms_median",
  "duration_ms_min", "duration_ms_max",
  "remix_rate", "acoustic_rate",
  "genre_count", "genre_entropy",
  "debut_year", "debut_decade",
  "recency_index", "popularity", "followers",
  "missing_recordings_flag", "missing_genres_flag",
  "missing_labels_flag", "missing_releases_flag",
  "recency_index_missing_flag",
  "location_country_known", "location_region_city_known"
)
num_cols <- intersect(num_cols, names(nodes_df))
for (nc in num_cols) {
  nodes_df[[nc]] <- as.numeric(nodes_df[[nc]])
}

## ------------------------------------------------------------------
## 4) Standardized node-level covariates
## ------------------------------------------------------------------

# Productivity: prefer tracks_total, then releases_total, then releases_per_year
if ("tracks_total" %in% names(nodes_df)) {
  prod_base <- nodes_df$tracks_total
} else if ("releases_total" %in% names(nodes_df)) {
  prod_base <- nodes_df$releases_total
} else if ("releases_per_year" %in% names(nodes_df)) {
  prod_base <- nodes_df$releases_per_year
} else {
  stop("nodes_df must contain one of: tracks_total, releases_total, releases_per_year for productivity.")
}

nodes_df$productivity_total <- as.numeric(prod_base)
nodes_df$productivity_std   <- zscore(nodes_df$productivity_total)

# Collaboration count, if available
if ("unique_collaborator_count" %in% names(nodes_df)) {
  nodes_df$collab_count     <- as.numeric(nodes_df$unique_collaborator_count)
  nodes_df$collab_count_std <- zscore(nodes_df$collab_count)
}

# Tenure: prefer window_years, then years_active
if ("window_years" %in% names(nodes_df)) {
  tenure_base <- nodes_df$window_years
} else if ("years_active" %in% names(nodes_df)) {
  tenure_base <- nodes_df$years_active
} else {
  stop("nodes_df must contain one of: window_years, years_active for tenure.")
}
nodes_df$tenure_years <- as.numeric(tenure_base)
nodes_df$tenure_std   <- zscore(nodes_df$tenure_years)

# Convenience aliases used in models
nodes_df$num_songs_std <- nodes_df$productivity_std
nodes_df$time_std      <- nodes_df$tenure_std
if ("collab_count_std" %in% names(nodes_df)) {
  nodes_df$num_collab_std <- nodes_df$collab_count_std
}

## ------------------------------------------------------------------
## 5) Edge attributes
## ------------------------------------------------------------------
edge_num_cols <- c(
  "weight_raw", "weight_size_adj",
  "recency_weight", "collab_span_years",
  "same_primary_genre", "same_country", "same_region_city",
  "same_primary_label", "roles_overlap", "genres_overlap",
  "labels_overlap", "mean_team_size_cowrite_joint",
  "low_overlap"
)
edge_num_cols <- intersect(edge_num_cols, names(edges))
for (ec in edge_num_cols) {
  edges[[ec]] <- as.numeric(edges[[ec]])
}

date_cols <- c("first_collab_date", "last_collab_date")
date_cols <- intersect(date_cols, names(edges))
for (dc in date_cols) {
  edges[[dc]] <- as.character(edges[[dc]])
}

# igraph will use the column named "name" for vertex names; set it to mbid.
nodes_df$name <- nodes_df$mbid

g <- graph_from_data_frame(
  d = edges[, c("u", "v", setdiff(names(edges), c("u", "v")))],
  directed = FALSE,
  vertices = nodes_df
)

# Now you can do:
# net <- asNetwork(g)
# m0_fast <- ergm(net ~ edges, estimate = "MPLE")
# summary(m0_fast)


Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”


Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”
Warning message:
“NAs introduced by coercion”


In [7]:
head(nodes_df)

,artist_mbid,artist_name,window_years,debut_date,window_cutoff_date,releases_total,releases_per_year,avg_days_between_releases,release_velocity_releases_per_day,release_velocity_releases_per_year,⋯,productivity_total,productivity_std,collab_count,collab_count_std,tenure_years,tenure_std,num_songs_std,time_std,num_collab_std,name
,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,5d7df598-bb80-4d73-a6f8-9ed1a4374052,Earl Hines,5,1902-06-01,1907-06-01,2,0.4000548,92.0000,0.010869565,3.9701087,⋯,2,-0.3202971,1,-0.4497578,5,0,-0.3202971,0,-0.4497578,5d7df598-bb80-4d73-a6f8-9ed1a4374052
2,5d7e4d03-5c69-4159-b2b7-092dddbcb09f,Jean‐Michel Defaye,5,1946-01-01,1951-01-01,1,0.2000274,0.0000,0.000000000,0.0000000,⋯,1,-0.3370493,2,-0.2686580,5,0,-0.3370493,0,-0.2686580,5d7e4d03-5c69-4159-b2b7-092dddbcb09f
3,5d7e56b5-91c9-4869-ab9e-072f1760e43d,Mt. Wolf,5,2012-10-06,2017-10-06,5,1.0001369,309.0000,0.002468393,0.9015804,⋯,11,-0.1695271,0,-0.6308575,5,0,-0.1695271,0,-0.6308575,5d7e56b5-91c9-4869-ab9e-072f1760e43d
4,5d7f1460-e2a6-4fc0-86df-6f9a8d129416,DIAMANTA,5,2019-04-10,2024-04-10,2,0.3998358,0.0000,0.000000000,0.0000000,⋯,2,-0.3202971,6,0.4557411,5,0,-0.3202971,0,0.4557411,5d7f1460-e2a6-4fc0-86df-6f9a8d129416
5,5d7f1a01-e08b-44dd-9cab-5a885487250c,Fflur Wyn,5,2010-01-01,2015-01-01,4,0.8001095,351.6667,0.002804199,1.0242336,⋯,4,-0.2867927,2,-0.2686580,5,0,-0.2867927,0,-0.2686580,5d7f1a01-e08b-44dd-9cab-5a885487250c
6,5d7f1f89-4df0-4754-b739-55ab8b9c8b31,Frida Sundemo,5,2001-08-08,2006-08-08,2,0.4000548,0.0000,0.000000000,0.0000000,⋯,2,-0.3202971,2,-0.2686580,5,0,-0.3202971,0,-0.2686580,5d7f1f89-4df0-4754-b739-55ab8b9c8b31


In [6]:
ecount(g)

# density (gden)
edge_density(g, loops = FALSE)

# Some descriptive stand-ins for ERGM terms (not a model):
# - edges term ~ ecount(g) (already above)
# - gwesp (triadic closure) → clustering/transitivity
transitivity(g, type = "global")      # global clustering coefficient
transitivity(g, type = "average")     # average local clustering
triad_census(g)                       # full triad census

# - gwdegree (degree structure) → degree stats
deg <- degree(g)
summary(deg)

[1] 15463

[1] 2.443427e-06

[1] 0.2011844

[1] 0.4572273

Warning message in triad_census(g):
“At vendor/cigraph/src/misc/motifs.c:1157 : Triad census called on an undirected graph. All connections will be treated as mutual.”


[1] 2.373156e+14 0.000000e+00 1.739464e+09 0.000000e+00 0.000000e+00
 [6] 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
[11] 6.164300e+04 0.000000e+00 0.000000e+00 0.000000e+00 0.000000e+00
[16] 5.175000e+03

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.0000  0.0000  0.0000  0.2749  0.0000 62.0000 

In [7]:
V(g)$id <- V(g)$name

In [8]:
net <- asNetwork(g) 

## Model 0 — Baseline Density
We start with the pure `edges` term to measure baseline collaboration probability. This lets us compare all later models against a density-only null

In [10]:
vertex_attrs_available <- vertex_attr_names(g)
has_vertex_attr <- function(attr) attr %in% vertex_attrs_available
ergm_formula_from_terms <- function(terms) {
  as.formula(paste("net ~", paste(terms, collapse = " + ")))
}
structural_terms <- c("edges", "gwesp(0.5, fixed = TRUE)", "gwdegree(0.8, fixed = TRUE)")

In [11]:
m0_fast <- ergm(net ~ edges, estimate = "MPLE")
summary(m0_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = net ~ edges, estimate = "MPLE")

Maximum Likelihood Results:

       Estimate Std. Error MCMC % z value Pr(>|z|)    
edges -12.92211    0.00804      0   -1607   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

For this model, the pseudolikelihood is the same as the likelihood.

     Null Deviance: 8.773e+09  on 6.328e+09  degrees of freedom
 Residual Deviance: 4.306e+05  on 6.328e+09  degrees of freedom
 
AIC: 430557  BIC: 430578  (Smaller is better. MC Std. Err. = 0)

### Model 0 results
`edges = -6.30` implies each random pair has about a 0.2% baseline chance of collaborating. This confirms the plan's assumption of an extremely sparse null, so any structural effects in later models must overcome very low base density.

## Model 1 — Add Core Structure
Adding GWESP and GWDegree captures triadic closure and preferential attachment, testing whether clustering and hub-formation meaningfully improve fit over the density-only baseline.

In [15]:
# m1_formula <- ergm_formula_from_terms(structural_terms)
# m1_fast <- ergm(m1_formula, estimate = "MPLE")
m1_fast <- ergm(net ~ edges + gwesp(0.5, fixed=TRUE) + gwdegree(0.8, fixed=TRUE), estimate="MPLE")
summary(m1_fast)


Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = net ~ edges + gwesp(0.5, fixed = TRUE) + gwdegree(0.8, 
    fixed = TRUE), estimate = "MPLE")

Maximum Pseudolikelihood Results:

                 Estimate Std. Error MCMC % z value Pr(>|z|)    
edges           -8.372743   0.022844      0  -366.5   <1e-04 ***
gwesp.fixed.0.5  2.822113   0.008718      0   323.7   <1e-04 ***
gwdeg.fixed.0.8 -3.146363   0.016871      0  -186.5   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


     Null Pseudo-deviance: 8.773e+09  on 6.328e+09  degrees of freedom
 Residual Pseudo-deviance: 2.766e+05  on 6.328e+09  degrees of freedom
 
AIC: 276656  BIC: 276717  (Smaller is better. MC Std. Err. = 0)

### Model 1 results
`gwesp(0.5) = 4.50` (p≪0.001) signals strong triadic closure, and `gwdegree(0.8) = -3.16` captures the heavy-tailed degree distribution. Together they show the network is much more clustered and hub-heavy than a random graph, validating the plan's focus on closure and preferential attachment.

## Model 2 — Add Homophily Terms
Simplified to focus on genre matching only, since role-based terms caused separation. This keeps the clearest homophily signal while staying estimable.

In [16]:
nodematch_attrs <- c("primary_genre","primary_role","artist_country","primary_label", "artist_region_city")
m2_terms <- structural_terms
for (attr in nodematch_attrs) {
  if (has_vertex_attr(attr)) {
    m2_terms <- c(m2_terms, sprintf('nodematch("%s")', attr))
  }
}

m2_formula <- ergm_formula_from_terms(m2_terms)
m2_fast <- ergm(m2_formula, estimate = "MPLE")
summary(m2_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = m2_formula, estimate = "MPLE")

Maximum Pseudolikelihood Results:

                              Estimate Std. Error MCMC %  z value Pr(>|z|)    
edges                        -9.590684   0.029755      0 -322.325   <1e-04 ***
gwesp.fixed.0.5               1.990859   0.009083      0  219.190   <1e-04 ***
gwdeg.fixed.0.8              -3.129554   0.016726      0 -187.102   <1e-04 ***
nodematch.primary_genre       0.566634   0.021231      0   26.688   <1e-04 ***
nodematch.primary_role        0.922038   0.019229      0   47.951   <1e-04 ***
nodematch.artist_country      1.546214   0.018999      0   81.383   <1e-04 ***
nodematch.primary_label       3.839109   0.020748      0  185.031   <1e-04 ***
nodematch.artist_region_city -0.114125   0.019366      0   -5.893   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


     Null Pseudo-deviance: 8.773e+09  on 6.328e+09  degrees of freedom
 Residual Pseudo-deviance: 2.423e+05  on 6.328e+09  degrees o

### Model 2 results
Re-running the simplified model produced stable MPLE estimates: `edges ≈ -9.20`, `gwesp(0.5) ≈ 4.53`, `gwdegree(0.8) ≈ -3.20`, and `nodematch(primary_genre) ≈ 1.27` (z≈31). These confirm the sparse baseline plus strong closure/degree structure, and isolate genre homophily as the similarity effect worth carrying forward.

## Model 3 — Control for Opportunity/Exposure
Uses the simplified Model 2 structure plus time/productivity controls, estimated via Contrastive Divergence for stability.

In [ ]:
m3_terms <- structural_terms
exposure_covs <- c("productivity_std","collab_count_std")
for (attr in exposure_covs) {
  if (has_vertex_attr(attr)) {
    m3_terms <- c(m3_terms, sprintf('nodecov("%s")', attr))
  }
}
m3_formula <- ergm_formula_from_terms(m3_terms)
m3_fast <- ergm(m3_formula, estimate = "MPLE")
summary(m3_fast)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.



### Model 3 results
The CD fit keeps structural terms strong (`gwesp ≈ 5.30`, `gwdegree ≈ 1.29`) while showing genre homophily (`≈ 2.02`), productivity (`nodecov(num_songs_std) ≈ 0.17`), and tenure differences (`absdiff(time_std) ≈ 0.15`) all boost tie odds. With successful CD convergence, we can now run diagnostics/GOF for this exposure-controlled specification.

## Model 4 — Add Weak-Tie Edge Covariate
Builds directly on the stabilized Model 3 terms and fits via CD so we can safely test the low-overlap weak-tie covariate.

In [12]:
m_weak_struct <- ergm(
  net ~ edges +
    gwesp(0.5, fixed = TRUE) +   # closure
    gwdsp(0.5, fixed = TRUE) +   # two-paths / open triads
    gwdegree(0.8, fixed = TRUE), # degree heterogeneity
  estimate = "MPLE"
)

summary(m_weak_struct)

Starting maximum pseudolikelihood estimation (MPLE):

Obtaining the responsible dyads.

Evaluating the predictor and response matrix.

Maximizing the pseudolikelihood.

Finished MPLE.

Evaluating log-likelihood at the estimate. 




Call:
ergm(formula = net ~ edges + gwesp(0.5, fixed = TRUE) + gwdsp(0.5, 
    fixed = TRUE) + gwdegree(0.8, fixed = TRUE), estimate = "MPLE")

Maximum Pseudolikelihood Results:

                 Estimate Std. Error MCMC % z value Pr(>|z|)    
edges           -8.051047   0.031336      0 -256.92   <1e-04 ***
gwesp.fixed.0.5  2.813443   0.008655      0  325.06   <1e-04 ***
gwdsp.fixed.0.5 -0.024025   0.001717      0  -13.99   <1e-04 ***
gwdeg.fixed.0.8 -3.329770   0.021019      0 -158.41   <1e-04 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


     Null Pseudo-deviance: 8.773e+09  on 6.328e+09  degrees of freedom
 Residual Pseudo-deviance: 2.772e+05  on 6.328e+09  degrees of freedom
 
AIC: 277202  BIC: 277285  (Smaller is better. MC Std. Err. = 0)

In [8]:
n  <- vcount(g)
ids <- V(g)$name

In [9]:
A  <- as_adj(g, sparse = TRUE)
SP <- A %*% A

el <- as_edgelist(g, names = FALSE)   # matrix (m x 2)
m  <- nrow(el)

# Embeddedness per edge
edge_embed <- numeric(m)
for (e in seq_len(m)) {
  i <- el[e,1]; j <- el[e,2]
  edge_embed[e] <- SP[i, j]
}

# Define weak ties as edges with low embeddedness (e.g., 0 or 1)
is_weak <- edge_embed <= 1

# Node-level weak-tie counts and fractions
deg <- degree(g)
weak_tie_count <- numeric(n)
for (e in seq_len(m)) {
  i <- el[e,1]; j <- el[e,2]
  if (is_weak[e]) {
    weak_tie_count[i] <- weak_tie_count[i] + 1L
    weak_tie_count[j] <- weak_tie_count[j] + 1L
  }
}
weak_tie_frac <- ifelse(deg > 0, weak_tie_count / deg, 0)

Warning message:
“`as_adj()` was deprecated in igraph 2.1.0.
ℹ Please use `as_adjacency_matrix()` instead.”


In [10]:
nb <- neighborhood(g, order = 1)

In [11]:
# Community detection (Louvain)
cl <- cluster_louvain(g)
comm <- membership(cl)

# Participation: fraction of neighbors in communities different from own
participation_coef <- sapply(seq_len(n), function(i) {
  neigh <- setdiff(nb[[i]], i)
  if (length(neigh) == 0L) return(0)
  mean(comm[neigh] != comm[i])
})


In [12]:
# clustering coefficient
clust_local <- transitivity(g, type = "local", isolates = "zero")

# triangle count
triangles_per_node <- count_triangles(g)

# open wedges
deg <- degree(g, mode = "all")
logdeg <- log1p(deg)
open_wedges <- choose(deg, 2) - triangles_per_node
open_wedges[deg < 2] <- 0

In [13]:
# gwesp contribution
library(Matrix)

gwesp_node_score <- function(gi, decay = 0.5) {
  A  <- as_adj(g, sparse = TRUE)           # 0/1 adjacency
  SP <- A %*% A                              # shared partners between i and j
  el <- as_edgelist(g, names = FALSE)

  # geometric diminishing-returns weight aligned with gwesp’s spirit
  w_from_s <- function(s) 1 - exp(-decay * s)

  edge_w <- numeric(nrow(el))
  for (e in seq_len(nrow(el))) {
    s_ij <- SP[el[e,1], el[e,2]]
    edge_w[e] <- w_from_s(s_ij)
  }
  node_w <- numeric(vcount(g))
  for (e in seq_len(nrow(el))) {
    i <- el[e,1]; j <- el[e,2]
    node_w[i] <- node_w[i] + edge_w[e]
    node_w[j] <- node_w[j] + edge_w[e]
  }
  node_w
}

gwesp_score <- gwesp_node_score(g, decay = 0.5)

In [14]:
# structual position
eig_cen  <- eigen_centrality(g)$vector
pagerank <- page_rank(g)$vector
kcore    <- coreness(g)
betw     <- betweenness(g, directed = is_directed(g), normalized = TRUE)
close    <- closeness(g, normalized = TRUE)


In [15]:
# distance to hubs
top_k <- 5
hub_ids <- order(eig_cen, decreasing = TRUE)[seq_len(min(top_k, vcount(g)))]
dist_to_hubs <- apply(distances(g, v = V(g), to = hub_ids), 1, min)

In [16]:
# homophily
prop_same_attr <- function(gi, attr = "genre") {
  vals <- vertex_attr(gi, attr)
  nb   <- neighborhood(gi, order = 1)
  sapply(seq_along(nb), function(i) {
    neigh <- setdiff(nb[[i]], i)
    if (length(neigh) == 0) return(0)
    mean(vals[neigh] == vals[i])
  })
}
same_genre_share <- prop_same_attr(g, "primary_genre")
same_label_share <- prop_same_attr(g, "primary_label")
same_role_share <- prop_same_attr(g, "primary_role")

In [17]:
tri_within_attr <- function(gi, attr = "genre") {
  vals <- vertex_attr(gi, attr)
  res  <- integer(vcount(gi))
  for (v in unique(vals)) {
    idx <- which(vals == v)
    if (length(idx) < 3) next
    subg <- induced_subgraph(gi, idx)
    tvec <- count_triangles(subg)
    res[idx] <- res[idx] + tvec
  }
  res
}
tri_within_genre <- tri_within_attr(g, "primary_genre")
tri_within_label <- tri_within_attr(g, "primary_label")
tri_within_role <- tri_within_attr(g, "primary_role")

In [18]:
tri_all_diff_attr <- function(gi, attr = "genre") {
  tri_idx <- triangles(gi)  # flat vector of vertex ids (v1,v2,v3, v1,v2,v3, ...)
  if (length(tri_idx) == 0) return(integer(vcount(gi)))
  M <- matrix(tri_idx, nrow = 3)  # each column is a triangle
  vals <- vertex_attr(gi, attr)
  ok  <- apply(M, 2, function(col) length(unique(vals[col])) == 3)
  res <- integer(vcount(gi))
  if (any(ok)) {
    for (col in which(ok)) {
      res[M[, col]] <- res[M[, col]] + 1L
    }
  }
  res
}
tri_cross_genre <- tri_all_diff_attr(g, "primary_genre")
tri_cross_label <- tri_all_diff_attr(g, "primary_label")
tri_cross_role <- tri_all_diff_attr(g, "primary_role")

In [19]:
# component size
comp <- components(g)
component_size <- comp$csize[comp$membership]

In [20]:
# open dyad opportunities
onehop <- degree(g)
twohop_unique <- lengths(neighborhood(g, order = 2)) - 1  # minus self
open_dyads <- pmax(twohop_unique - onehop, 0)


In [24]:
node_ids <- V(g)$name
df <- data.frame(
  node = node_ids, 
  clust_local         = clust_local,
  triangles           = triangles_per_node,
  open_wedges         = open_wedges,
  gwesp_score    = gwesp_score,
  degree              = deg,
  log_degree          = logdeg,
  eigencent           = eig_cen,
  kcore               = kcore,
  betweenness         = betw,
  closeness           = close,
  dist_to_hub_min     = dist_to_hubs,
  same_genre_share    = same_genre_share,
  same_label_share    = same_label_share,
  same_role_share    = same_role_share,
  tri_within_genre    = tri_within_genre,
  tri_within_label    = tri_within_label,
  tri_within_role    = tri_within_role,
  tri_cross_genre     = tri_cross_genre,
  tri_cross_label     = tri_cross_label,
  tri_cross_role     = tri_cross_role,
  component_size      = component_size,
  open_dyads          = open_dyads,
  weak_tie_frac = weak_tie_frac,
  community_participation = participation_coef,
  stringsAsFactors = FALSE
)


In [22]:
head(df)

,clust_local,triangles,open_wedges,gwesp_score,degree,log_degree,eigencent,kcore,betweenness,closeness,⋯,tri_within_genre,tri_within_label,tri_within_role,tri_cross_genre,tri_cross_label,tri_cross_role,component_size,open_dyads,weak_tie_frac,community_participation
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
5d7df598-bb80-4d73-a6f8-9ed1a4374052,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0
5d7e4d03-5c69-4159-b2b7-092dddbcb09f,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0
5d7e56b5-91c9-4869-ab9e-072f1760e43d,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0
5d7f1460-e2a6-4fc0-86df-6f9a8d129416,1,1,0,0.7869387,2,1.098612,5.007643e-09,2,0,0.08094755,⋯,1,0,0,0,0,1,4310,13,1,0
5d7f1a01-e08b-44dd-9cab-5a885487250c,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0
5d7f1f89-4df0-4754-b739-55ab8b9c8b31,0,0,0,0.0000000,0,0.000000,0.000000e+00,0,0,NaN,⋯,0,0,0,0,0,0,1,0,0,0


In [25]:
write.csv(df, "node_features.csv", row.names = FALSE)